# Parent-Child Chunks
> Split documents into large parent chunks and small child chunks for
> fine-grained retrieval with full-context expansion.

`ParentChildChunker` creates a two-level hierarchy: large "parent" chunks
for context, and small "child" chunks for precise matching. At retrieval
time, `ParentExpander` maps matched children back to their parent.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.ingestion.chunkers import ParentChildChunker
from anchor.ingestion import ParentExpander

## Prepare a Document
We use a multi-section document to see how parent and child chunks align.

In [ ]:
document = (
    "Anchor is a framework for building context-aware AI applications. "
    "It provides tools for memory management, retrieval-augmented generation, "
    "and intelligent context assembly. The framework is designed to be modular, "
    "allowing developers to pick and choose the components they need. "
    "Each component can be used independently or combined into full pipelines.\n\n"
    "The ingestion module handles converting raw documents into structured "
    "context items. It supports multiple chunking strategies, each optimized "
    "for different content types. Text is split into overlapping chunks to "
    "preserve context at chunk boundaries. Parsers convert raw bytes into text "
    "before chunking begins.\n\n"
    "The retrieval module searches a vector store for relevant context items. "
    "It supports dense retrieval, sparse retrieval, and hybrid approaches. "
    "Results are scored and ranked before being added to the context window. "
    "Rerankers can further refine the ordering of retrieved items.\n\n"
    "The memory module manages conversation history and long-term knowledge. "
    "It supports sliding windows, summary buffers, and graph-based memory. "
    "Eviction policies control which turns are dropped when the token budget "
    "is exceeded. Decay strategies model how memories fade over time."
)

print(f"Document length: {len(document)} characters")

## Create the Parent-Child Chunker
Parent chunks are large (500 chars), child chunks are small (100 chars).
The overlap ensures no information is lost at boundaries.

In [ ]:
chunker = ParentChildChunker(
    parent_chunk_size=500,
    child_chunk_size=100,
    overlap=20,
)

print(f"Parent chunk size: {chunker.parent_chunk_size}")
print(f"Child chunk size:  {chunker.child_chunk_size}")
print(f"Overlap:           {chunker.overlap}")

## Chunk the Document
The chunker returns a structure containing both parent and child chunks
with their mapping.

In [ ]:
chunks = chunker.chunk(document)

print(f"Total chunk objects returned: {len(chunks)}")
print(f"\nChunk details:")
for i, chunk in enumerate(chunks):
    print(f"  [{i}] ({len(chunk)} chars): {chunk[:50]}...")

## Examine the Hierarchy
Let's manually create parent and child chunks to understand the hierarchy.

In [ ]:
# Simulate parent/child relationship
parent_size = 500
child_size = 100

# Create parent chunks
parents = []
for start in range(0, len(document), parent_size):
    parents.append(document[start:start + parent_size])

print(f"Parent chunks: {len(parents)}")
for i, parent in enumerate(parents):
    print(f"  Parent [{i}] ({len(parent)} chars): {parent[:50]}...")

print()

# Create child chunks from first parent
first_parent = parents[0]
children = []
for start in range(0, len(first_parent), child_size - 20):
    children.append(first_parent[start:start + child_size])

print(f"Children of Parent [0]: {len(children)}")
for i, child in enumerate(children):
    print(f"  Child [{i}] ({len(child)} chars): {child[:40]}...")

## Parent Expander Concept
`ParentExpander` maps child chunk matches back to their parent chunk
for full-context retrieval. In production, you provide a child retriever
and a parent store.

In [ ]:
# Simulate the parent expansion workflow

# 1. Build a mock parent store (parent_id -> parent_text)
parent_store = {}
child_to_parent = {}

for p_idx, parent in enumerate(parents):
    parent_id = f"parent-{p_idx}"
    parent_store[parent_id] = parent
    # Create children for this parent
    for c_idx in range(0, len(parent), child_size - 20):
        child_text = parent[c_idx:c_idx + child_size]
        child_id = f"child-{p_idx}-{c_idx}"
        child_to_parent[child_id] = parent_id

print(f"Parent store: {len(parent_store)} parents")
print(f"Child-to-parent map: {len(child_to_parent)} children")

In [ ]:
# 2. Simulate a child match and expand to parent
matched_child_id = list(child_to_parent.keys())[3]  # pick one
matched_parent_id = child_to_parent[matched_child_id]
expanded_text = parent_store[matched_parent_id]

print(f"Matched child:  {matched_child_id}")
print(f"Parent ID:      {matched_parent_id}")
print(f"Expanded text:  ({len(expanded_text)} chars)")
print(f"  {expanded_text[:80]}...")

## ParentExpander API
In a real application, `ParentExpander` wraps this lookup pattern.

In [ ]:
# ParentExpander usage pattern:
#
#   expander = ParentExpander(
#       retriever=child_retriever,   # retrieves matching child chunks
#       parent_store=parent_store,    # maps child -> parent for expansion
#   )
#
#   # At query time:
#   results = expander.retrieve(query="What is Anchor?")
#   # Returns parent chunks that contain matching children

print("ParentExpander workflow:")
print("  1. Query arrives")
print("  2. Child retriever finds matching small chunks")
print("  3. ParentExpander maps children to parent chunks")
print("  4. Full parent context is returned for the pipeline")
print()
print("Benefits:")
print("  - Small child chunks give precise similarity matching")
print("  - Large parent chunks provide full surrounding context")
print("  - Best of both worlds: precision + context")

## Choosing Chunk Sizes
The ratio between parent and child sizes controls the trade-off
between precision and context.

In [ ]:
configs = [
    {"parent": 256,  "child": 64,  "overlap": 10},
    {"parent": 512,  "child": 128, "overlap": 20},
    {"parent": 1024, "child": 256, "overlap": 50},
    {"parent": 2048, "child": 512, "overlap": 100},
]

print(f"{'Parent':>8} {'Child':>8} {'Overlap':>8} {'Ratio':>8} {'Use Case'}")
print("-" * 60)
for cfg in configs:
    ratio = cfg['parent'] / cfg['child']
    use_case = (
        "FAQ / short docs" if cfg['parent'] <= 256
        else "General prose" if cfg['parent'] <= 512
        else "Technical docs" if cfg['parent'] <= 1024
        else "Long-form content"
    )
    print(f"{cfg['parent']:>8} {cfg['child']:>8} {cfg['overlap']:>8} "
          f"{ratio:>7.1f}x {use_case}")

## Key Takeaways
- `ParentChildChunker` creates a two-level chunk hierarchy
- Small child chunks enable precise similarity matching
- Large parent chunks provide full surrounding context
- `ParentExpander` maps matched children back to their parent at retrieval time
- Tune the parent/child size ratio based on your content type and query patterns

**Back to:** [Ingestion README](README.md)